# Building a Basic AI Knowledge Assistant with Memory

**Mid-term Project (Week 3)** — Retrieval-Augmented Generation with conversational memory.

## Rubric checklist

1. **Data Integration** — PDF + Google Sheet → structured text
2. **Embedding & Indexing** — HuggingFace embeddings stored in **Qdrant Cloud**
3. **Question Answering Pipeline** — retrieve relevant chunks, generate answers with an LLM
4. **Memory Integration** — full chat buffer (+ optional summary memory)
5. **Testing & Evaluation** — demo questions with source inspection

---

## Kaggle setup (required before running)

1. **Settings**: GPU **T4**, **Internet ON**
2. **Dataset**: attach your PDF and set `PDF_PATH` in the config cell
3. **Google Sheet**: publish/share the sheet and set `SHEET_CSV_URL` to the CSV export URL
4. **Secrets** (Add-ons → Secrets): add `QDRANT_URL` and `QDRANT_API_KEY` — **never hardcode the API key**
5. **First run**: keep `RECREATE_COLLECTION = True` to build the Qdrant collection
6. **Later runs**: set `RECREATE_COLLECTION = False` to query without re-indexing

## Configuration

In [2]:
import os

PDF_PATH = "/kaggle/input/datasets/ahmednagah031220/cv-lectures/Lectures/Lec 1 -Overview of Computer Vision and Pattern Recognition.pdf"
SHEET_CSV_URL = "/kaggle/input/datasets/ahmednagah031220/egypt-houses-prices/Egypt_Houses_Price.csv"

# Qdrant Cloud — load from Kaggle Secrets (Add-ons → Secrets)
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF-TH_token")
QDRANT_API_KEY = user_secrets.get_secret("QDRANT_API_KEY")
QDRANT_URL = user_secrets.get_secret("QDRANT_URL")
QDRANT_COLLECTION = "knowledge_assistant"
RECREATE_COLLECTION = False  # True on first run; False to reuse existing indexed data

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
CHUNK_SIZE, CHUNK_OVERLAP = 800, 150
TOP_K = 3
USE_SUMMARY_MEMORY = True  # set True to enable optional summary memory
SUMMARY_BATCH, KEEP_RECENT = 3, 2

## Install dependencies

In [3]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-qdrant \
    qdrant-client sentence-transformers transformers accelerate pypdf pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires request

## Imports

In [4]:
import pandas as pd
import torch
from pathlib import Path

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

from transformers import AutoTokenizer, AutoModelForCausalLM

/tmp/ipykernel_104/3440491992.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 1. Data Integration

Load content from a PDF and a Google Sheet (CSV export), converting both into LangChain `Document` objects.

In [5]:
def load_sheet_documents(csv_url: str) -> list:
    """Convert each Google Sheet row into a structured text Document."""
    df = pd.read_csv(csv_url)
    return [
        Document(
            page_content=" | ".join(f"{c}: {row[c]}" for c in df.columns),
            metadata={"source": "google_sheet", "row": int(idx)},
        )
        for idx, row in df.iterrows()
    ]


def load_pdf_documents(pdf_path: str) -> list:
    """Load PDF pages and tag metadata."""
    if not Path(pdf_path).exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")
    docs = PyPDFLoader(pdf_path).load()
    for doc in docs:
        doc.metadata["source"] = "pdf"
    return docs


def load_all_documents(pdf_path: str, sheet_csv_url: str) -> list:
    pdf_docs = load_pdf_documents(pdf_path)
    sheet_docs = load_sheet_documents(sheet_csv_url)
    return pdf_docs + sheet_docs

In [6]:
raw_documents = load_all_documents(PDF_PATH, SHEET_CSV_URL)

pdf_count = sum(1 for d in raw_documents if d.metadata.get("source") == "pdf")
sheet_count = sum(1 for d in raw_documents if d.metadata.get("source") == "google_sheet")

print(f"Loaded {len(raw_documents)} documents ({pdf_count} PDF pages, {sheet_count} sheet rows)")
print("\nSample sheet row:")
print(next(d.page_content for d in raw_documents if d.metadata.get("source") == "google_sheet")[:300])
print("\nSample PDF excerpt:")
print(next(d.page_content for d in raw_documents if d.metadata.get("source") == "pdf")[:300])

Loaded 27432 documents (71 PDF pages, 27361 sheet rows)

Sample sheet row:
Type: Duplex | Price: 4000000 | Bedrooms: 3.0 | Bathrooms: 3.0 | Area: 400.0 | Furnished: No | Level: 7 | Compound: Unknown | Payment_Option: Cash | Delivery_Date: Ready to move | Delivery_Term: Finished | City: Nasr City

Sample PDF excerpt:
Department of Computer Science & Engineering 
Instructor  : Dr. Ahmed Gomaa
Spring, 2026
CSE-429 
Computer Vision and pattern recognition


## 2. Embedding & Indexing (Qdrant Cloud)

Split documents into chunks, embed with HuggingFace, and store vectors in Qdrant.

In [7]:
if not QDRANT_URL or not QDRANT_API_KEY:
    raise ValueError(
        "Set QDRANT_URL and QDRANT_API_KEY via Kaggle Secrets (Add-ons → Secrets)."
    )

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = text_splitter.split_documents(raw_documents)
print(f"Split into {len(chunks)} chunks")

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

if RECREATE_COLLECTION:
    if qdrant_client.collection_exists(QDRANT_COLLECTION):
        qdrant_client.delete_collection(QDRANT_COLLECTION)
        print(f"Deleted existing collection: {QDRANT_COLLECTION}")

    vectordb = QdrantVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
        collection_name=QDRANT_COLLECTION,
    )
    print(f"Indexed {len(chunks)} chunks into Qdrant collection '{QDRANT_COLLECTION}'")
else:
    if not qdrant_client.collection_exists(QDRANT_COLLECTION):
        raise ValueError(
            f"Collection '{QDRANT_COLLECTION}' not found. Set RECREATE_COLLECTION = True first."
        )
    vectordb = QdrantVectorStore.from_existing_collection(
        embedding=embeddings,
        collection_name=QDRANT_COLLECTION,
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
    )
    print(f"Loaded existing Qdrant collection '{QDRANT_COLLECTION}'")

Split into 27433 chunks


/tmp/ipykernel_104/2017530733.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Deleted existing collection: knowledge_assistant
Indexed 27433 chunks into Qdrant collection 'knowledge_assistant'


In [8]:
# Sanity check: retrieve sample chunks
test_query = "What are stages of image processing"
sample_docs = vectordb.similarity_search(test_query, k=TOP_K)

print(f"Top {TOP_K} results for '{test_query}':\n")
for i, doc in enumerate(sample_docs, 1):
    print(f"--- Result {i} (source={doc.metadata.get('source')}) ---")
    print(doc.page_content[:250], "\n")

Top 3 results for 'What is the process of computer vision':

--- Result 1 (source=pdf) ---
How hard is computer vision?
37 

--- Result 2 (source=pdf) ---
Expected 
Outcome
 Process and analyze images using different computational techniques.
 Apply edge detection, segmentation, and feature extraction methods.
 Implement and evaluate CNNs for image classification tasks.
 Work with real-world image  

--- Result 3 (source=pdf) ---
GENERAL INFORMATION
CSE 429 – Computer Vision and pattern recognition
Recommended 
References
1. “Digital Image Processing” ,Rafael C. Gonzalez & Richard E. Woods, Third Edition, 2008
2. Computer Vision, Linda G. Shapiro and George Stockman, Prentice 



## 3. Load Generator LLM

Local HuggingFace model for answer generation (fits Kaggle T4 GPU).

In [9]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    trust_remote_code=True,
    device_map="auto",
    dtype="auto",
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model ready on {device}")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model ready on cuda


In [10]:
def get_response(messages, max_new_tokens: int = 256) -> str:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

## 4. Memory Integration

- **Buffer memory** (default): full conversation history in every prompt
- **Summary memory** (optional): compress older turns when `USE_SUMMARY_MEMORY = True`

In [11]:
conversation_history = []
conversation_summary = ""


def clear_memory():
    global conversation_history, conversation_summary
    conversation_history = []
    conversation_summary = ""
    print("Conversation memory cleared.")


def build_messages(history: list, summary: str, user_message: str) -> list:
    system_content = (
        "You are a helpful AI knowledge assistant. "
        "Answer using ONLY the retrieved context provided in the user message. "
        "If the context does not contain the answer, say you do not know. "
        "Use conversation history to handle follow-up questions."
    )

    messages = [{"role": "system", "content": system_content}]

    if USE_SUMMARY_MEMORY and summary.strip():
        messages.append({
            "role": "system",
            "content": f"Long-Term Memory:\n{summary}",
        })

    for turn in history:
        messages.append({"role": "user", "content": turn["user"]})
        messages.append({"role": "assistant", "content": turn["assistant"]})

    messages.append({"role": "user", "content": user_message})
    return messages

In [12]:
def summarize_history():
    global conversation_summary, conversation_history

    if not USE_SUMMARY_MEMORY:
        return
    if len(conversation_history) < (SUMMARY_BATCH + KEEP_RECENT):
        return

    history_to_summarize = conversation_history[:SUMMARY_BATCH]
    recent_history = conversation_history[SUMMARY_BATCH:]

    history_text = "\n".join(f"User: {turn['user']}" for turn in history_to_summarize)

    summary_prompt = [
        {
            "role": "system",
            "content": (
                "Extract ONLY important facts the user explicitly stated about themselves. "
                "Return concise bullet points. Do not invent facts."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Existing Memory:\n{conversation_summary}\n\n"
                f"New User Messages:\n{history_text}\n\n"
                "Merge and return the complete updated memory."
            ),
        },
    ]

    conversation_summary = get_response(summary_prompt, max_new_tokens=120)
    conversation_history = recent_history
    print("\nSummary memory updated.\n")

In [13]:
def rag_chat(user_message: str):
    docs = vectordb.similarity_search(user_message, k=TOP_K)
    context = "\n\n".join(doc.page_content for doc in docs)
    augmented = (
        f"Context from knowledge base:\n{context}\n\n"
        f"Question: {user_message}"
    )

    messages = build_messages(conversation_history, conversation_summary, augmented)
    response = get_response(messages)

    conversation_history.append({"user": user_message, "assistant": response})
    summarize_history()

    return response, docs


def show_chat(user_message: str):
    response, docs = rag_chat(user_message)
    print(f"User     : {user_message}")
    print(f"Assistant: {response}")
    print(f"Sources  : {[d.metadata.get('source') for d in docs]}")
    print()

## 5. Testing & Evaluation

Run structured tests to validate retrieval accuracy, answer relevance, and conversational memory.

### Test 1 — Factual question from Google Sheet

In [15]:
clear_memory()
show_chat("What is Computer Vision (CV)?")

Conversation memory cleared.
User     : What is Computer Vision (CV)?
Assistant: Computer Vision (CV) involves teaching computers to interpret and understand visual data from the world as a human would. It encompasses various methods and algorithms used to process and analyze images or videos for understanding their content. This field includes tasks such as object detection, image classification, scene understanding, and more.
Sources  : ['pdf', 'pdf', 'pdf']



In [16]:
show_chat("What is Computer Vision (CV)?")

User     : What is Computer Vision (CV)?
Assistant: Computer Vision (CV) is the field of study that enables computers to interpret and understand visual data from the world as a human would. This includes processing and analyzing images or videos for understanding their content, which can involve tasks like object detection, image classification, and scene understanding.
Sources  : ['pdf', 'pdf', 'pdf']



### Test 2 — Factual question from PDF

In [29]:
show_chat("What is data science?")

User     : What is data science?
Assistant: Based on the provided context, there seems to be no information about data science. The context appears to be related to a course titled "Computer Vision and Pattern Recognition" at a university. Therefore, I don't have the answer to your question about data science within the given context.
Sources  : ['pdf', 'pdf', 'pdf']



### Test 3 — Conversational memory (buffer)

In [17]:
show_chat("Hi! My name is Nagah and I study Computer Science.")
show_chat("What is my name?")
show_chat("What do you know about me so far?")

User     : Hi! My name is Nagah and I study Computer Science.
Assistant: Hello Nagah! It's great to meet you. Based on the context provided, it seems you are enrolled in CSE-429, a course on Computer Vision and Pattern Recognition at the Department of Computer Science & Engineering. The course is at the undergraduate level and has prerequisites including MTH 123/ ECE 223, Signals and Systems. The course meets on Wednesdays from 9:00 to 10:40 AM, with office hours to be determined. Attendance is expected to be at least 75%, and grading is based on a final exam worth 40%, a midterm exam worth 30%, and class work (assignments and projects) worth 30%.
Sources  : ['pdf', 'pdf', 'pdf']

User     : What is my name?
Assistant: Your name is Nagah.
Sources  : ['google_sheet', 'google_sheet', 'google_sheet']


Summary memory updated.

User     : What do you know about me so far?
Assistant: Based on the information provided, it appears you are interested in apartments and have preferences for loca

### Test 4 — Follow-up question (memory + re-retrieval)

In [18]:
show_chat("What are the applications of computer visoin?")
show_chat("Give me an example of computer vision applications")

User     : What are the applications of computer visoin?
Assistant: The context provided does not specifically list applications of computer vision. However, based on the general information given, computer vision has applications in various fields such as security systems, medical imaging, robotics, autonomous vehicles, and more. The examples mentioned are less than 7 years old and the field is very active with many new applications expected in the future.
Sources  : ['pdf', 'pdf', 'pdf']

User     : Give me an example of computer vision applications
Assistant: An example of a computer vision application is facial recognition technology used by companies like Amazon for unlocking devices or by retailers to track customer behavior in stores. Another example is autonomous driving systems that use computer vision to interpret visual information from cameras to make decisions about steering, acceleration, and braking.
Sources  : ['pdf', 'pdf', 'pdf']



### Test 5 — Source inspection

In [19]:
query = "What are applications of CV?"
retrieved = vectordb.similarity_search(query, k=TOP_K)

print(f"Query: {query}\n")
for i, doc in enumerate(retrieved, 1):
    print(f"[{i}] source={doc.metadata.get('source')} row={doc.metadata.get('row', 'n/a')}")
    print(doc.page_content[:400])
    print("-" * 60)

Query: What are applications of CV?

[1] source=pdf row=n/a
Computer vision in the real-world
• Most examples are less than 7 years old 
• Very active research area. Many new applications to come. 
• A website of computer vision industries maintained by Prof. 
David Lowe (UBC)
• Note: website is old but interesting
• Note: David Lowe retired and moved to Google 2015 to 2018
http://www.cs.ubc.ca/~lowe/vision.html
71
------------------------------------------------------------
[2] source=pdf row=n/a
Final Course Project:  Machine Learning for Some 
Kind of Application
7
• a
 • c
• b
• d
------------------------------------------------------------
[3] source=pdf row=n/a
Computer Vision
• Low Level Vision
– Measurements
– Enhancements
– Region segmentation
– Features
• Mid Level Vision
– Reconstruction
– Depth
– Motion Estimation
• High Level Vision
– Category detection
– Activity recognition
– Deep understandings
The car is in front of the pole
25
-----------------------------------------

### Optional — Interactive chat loop

Uncomment and run in Kaggle to chat interactively. Type `exit` to quit or `clear` to reset memory.

In [ ]:
clear_memory()
print("Knowledge Assistant ready. Type 'exit' to quit, 'clear' to reset memory.\n")
while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() == "exit":
        break
    if user_input.lower() == "clear":
        clear_memory()
        continue
    show_chat(user_input)